# Decorators — Senior Python Interview Practice\n\nFour interview-style exercises spanning implementation, trade-offs, and production concerns.\n\n**How to use:** attempt each prompt first, then run and critique the reference solution. Discuss trade-offs aloud as you would in a senior-level interview.


## 1. Timing decorator\n\n### Problem statement\nCreate a decorator that reports execution duration while preserving function metadata.


In [ ]:
from functools import wraps
from time import perf_counter

def timed(fn):
    @wraps(fn)
    def wrapped(*args, **kwargs):
        start = perf_counter()
        try: return fn(*args, **kwargs)
        finally: print(f'{fn.__name__}: {perf_counter() - start:.6f}s')
    return wrapped


### Complexity\n- **Time:** Function cost + O(1)\n- **Space:** O(1)\n\n### Interview tip\nUse `finally` so failed calls are measured too, and `wraps` for introspection.\n\n### Follow-up questions\n- How would you send this to structured telemetry instead of printing?


## 2. Authorization decorator\n\n### Problem statement\nWrite a decorator factory requiring a named permission on the first argument, a user object.


In [ ]:
from functools import wraps

def requires(permission):
    def decorate(fn):
        @wraps(fn)
        def wrapped(user, *args, **kwargs):
            if permission not in user.permissions:
                raise PermissionError(permission)
            return fn(user, *args, **kwargs)
        return wrapped
    return decorate


### Complexity\n- **Time:** O(1) average\n- **Space:** O(1)\n\n### Interview tip\nDiscuss where authorization belongs: decorators work well at a boundary, not everywhere.\n\n### Follow-up questions\n- How would you support resource-level permissions?


## 3. Result cache\n\n### Problem statement\nWrite a cache decorator for hashable positional and keyword arguments.


In [ ]:
from functools import wraps

def cached(fn):
    cache = {}
    @wraps(fn)
    def wrapped(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache: cache[key] = fn(*args, **kwargs)
        return cache[key]
    return wrapped


### Complexity\n- **Time:** O(1) average plus function cost\n- **Space:** O(number of unique calls)\n\n### Interview tip\nMention that mutable/unhashable inputs and cache eviction need a policy.\n\n### Follow-up questions\n- Add TTL, max size, invalidation, and thread safety.


## 4. Rate limit decorator\n\n### Problem statement\nImplement a simple per-process rate limit allowing `limit` calls per window.


In [ ]:
from collections import deque
from functools import wraps
from time import monotonic

def rate_limit(limit, window):
    calls = deque()
    def decorate(fn):
        @wraps(fn)
        def wrapped(*args, **kwargs):
            now = monotonic()
            while calls and calls[0] <= now - window: calls.popleft()
            if len(calls) >= limit: raise RuntimeError('rate limit exceeded')
            calls.append(now); return fn(*args, **kwargs)
        return wrapped
    return decorate


### Complexity\n- **Time:** O(1) amortized\n- **Space:** O(limit)\n\n### Interview tip\nUse monotonic time; wall clocks can move backward.\n\n### Follow-up questions\n- Why is this insufficient for multi-process or distributed services?
